In [ ]:
# 1. Verificacao de ambiente
import importlib
import importlib.metadata
import platform

packages = ["torch", "diffusers", "transformers", "accelerate", "safetensors", "PIL", "matplotlib"]
missing = [name for name in packages if importlib.util.find_spec(name) is None]

print(f"Python: {platform.python_version()}")
if missing:
    raise RuntimeError(
        "Pacotes ausentes: " + ", ".join(missing) +
        "\nInstale o ambiente antes de executar: pip install torch diffusers transformers accelerate safetensors pillow matplotlib"
    )

import torch
from diffusers import StableDiffusionPipeline

print(f"PyTorch: {torch.__version__}")
print(f"Diffusers: {importlib.metadata.version('diffusers')}")

cuda_available = torch.cuda.is_available()
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    device = "cuda"
    print(f"GPU: {gpu_name}")
    print(f"VRAM total: {vram_gb:.2f} GB")
else:
    device = "cpu"
    print("CUDA indisponivel: fallback controlado em CPU")
    print("AVISO: a execucao em CPU sera lenta; use apenas para validacao.")

In [ ]:
# 2. Baseline com StableDiffusionPipeline e DeepCache aplicado
from pathlib import Path
import time
import torch
from diffusers import StableDiffusionPipeline

candidates = [
    Path.cwd(),
    Path.cwd() / "deepcache-experiment",
    Path.cwd().parent / "deepcache-experiment",
]
PROJECT_DIR = next((p for p in candidates if (p / "data" / "prompts_geracao.txt").exists()), Path.cwd())
OUTPUT_DIR = PROJECT_DIR / "results" / "baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "runwayml/stable-diffusion-v1-5"
SEED = 42
WIDTH = 512
HEIGHT = 512
NUM_INFERENCE_STEPS = 20
GUIDANCE_SCALE = 7.5
CACHE_BRANCH_ID = 2
CACHE_INTERVAL = 3

dtype = torch.float16 if device == "cuda" else torch.float32
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe = pipe.to(device)

if device == "cuda":
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass
    pipe.enable_attention_slicing()

cache_state = {
    "enabled": False,
    "step_index": 0,
    "last_timestep": None,
    "update_cache": True,
}

class DeepCacheModule(torch.nn.Module):
    """Reutiliza a saida de um bloco da U-Net em passos intermediarios."""

    def __init__(self, module, state):
        super().__init__()
        self.module = module
        self.state = state
        self.cache = None

    def reset(self):
        self.cache = None

    def forward(self, *args, **kwargs):
        if not self.state["enabled"]:
            return self.module(*args, **kwargs)
        if self.state["update_cache"] or self.cache is None:
            result = self.module(*args, **kwargs)
            self.cache = result
            return result
        return self.cache

down_index = min(CACHE_BRANCH_ID, len(pipe.unet.down_blocks) - 1)
up_index = min(CACHE_BRANCH_ID, len(pipe.unet.up_blocks) - 1)
down_cache = DeepCacheModule(pipe.unet.down_blocks[down_index], cache_state)
up_cache = DeepCacheModule(pipe.unet.up_blocks[up_index], cache_state)
pipe.unet.down_blocks[down_index] = down_cache
pipe.unet.up_blocks[up_index] = up_cache

original_unet_forward = pipe.unet.forward

def unet_forward_with_cache(*args, **kwargs):
    timestep = kwargs.get("timestep", args[1] if len(args) > 1 else None)
    timestep_key = None
    if torch.is_tensor(timestep):
        timestep_key = timestep.detach().cpu().tolist()
    elif timestep is not None:
        timestep_key = str(timestep)

    if timestep_key != cache_state["last_timestep"]:
        cache_state["last_timestep"] = timestep_key
        cache_state["step_index"] += 1
    cache_state["update_cache"] = (
        cache_state["step_index"] % CACHE_INTERVAL == 1
    )
    return original_unet_forward(*args, **kwargs)

pipe.unet.forward = unet_forward_with_cache

def set_deepcache(enabled):
    cache_state["enabled"] = enabled
    cache_state["step_index"] = 0
    cache_state["last_timestep"] = None
    cache_state["update_cache"] = True
    down_cache.reset()
    up_cache.reset()

def seed_all(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"Modelo: {MODEL_ID}")
print(f"Dispositivo: {device}")
print(f"DeepCache: cache_branch_id={CACHE_BRANCH_ID}, cache_interval={CACHE_INTERVAL}")

In [ ]:
# 3. Geracao comparativa, tempos e grade visual
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

prompts_file = PROJECT_DIR / "data" / "prompts_geracao.txt"
prompts = [line.strip() for line in prompts_file.read_text(encoding="utf-8").splitlines() if line.strip()]
if not prompts:
    raise RuntimeError(f"Nenhum prompt encontrado em {prompts_file}")

results = []

def infer_one(prompt, deepcache_enabled):
    set_deepcache(deepcache_enabled)
    seed_all(SEED)
    generator = torch.Generator(device=device).manual_seed(SEED)

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()
    image = pipe(
        prompt=prompt,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        width=WIDTH,
        height=HEIGHT,
        generator=generator,
    ).images[0]
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    peak_vram_mb = None
    if device == "cuda":
        peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return image, elapsed, peak_vram_mb

for idx, prompt in enumerate(prompts, start=1):
    baseline_image, baseline_time, baseline_vram = infer_one(prompt, False)
    deepcache_image, deepcache_time, deepcache_vram = infer_one(prompt, True)

    baseline_path = OUTPUT_DIR / f"prompt_{idx:02d}_baseline.png"
    deepcache_path = OUTPUT_DIR / f"prompt_{idx:02d}_deepcache.png"
    baseline_image.save(baseline_path)
    deepcache_image.save(deepcache_path)

    speedup = baseline_time / deepcache_time if deepcache_time > 0 else float("inf")
    results.append({
        "prompt": prompt,
        "baseline_time": baseline_time,
        "deepcache_time": deepcache_time,
        "speedup": speedup,
        "baseline_vram_mb": baseline_vram,
        "deepcache_vram_mb": deepcache_vram,
    })

    print(f"Prompt {idx}/{len(prompts)}")
    print(f"  Sem Cache: {baseline_time:.3f} s" + (f" | pico VRAM: {baseline_vram:.1f} MB" if baseline_vram else ""))
    print(f"  DeepCache: {deepcache_time:.3f} s" + (f" | pico VRAM: {deepcache_vram:.1f} MB" if deepcache_vram else ""))
    print(f"  Speedup: {speedup:.2f}x")

    canvas = Image.new("RGB", (WIDTH * 2, HEIGHT + 70), "white")
    canvas.paste(baseline_image.resize((WIDTH, HEIGHT)), (0, 70))
    canvas.paste(deepcache_image.resize((WIDTH, HEIGHT)), (WIDTH, 70))
    draw = ImageDraw.Draw(canvas)
    draw.text((10, 8), f"Sem Cache | {baseline_time:.3f} s", fill="black")
    draw.text((WIDTH + 10, 8), f"DeepCache | {deepcache_time:.3f} s", fill="black")
    draw.text((10, 38), prompt[:110], fill="black")
    display(canvas)

plt.figure(figsize=(12, 4))
labels = [f"P{i}" for i in range(1, len(results) + 1)]
baseline_times = [r["baseline_time"] for r in results]
deepcache_times = [r["deepcache_time"] for r in results]
x = range(len(labels))
width = 0.35
plt.bar([i - width / 2 for i in x], baseline_times, width, label="Sem Cache")
plt.bar([i + width / 2 for i in x], deepcache_times, width, label="DeepCache")
plt.xticks(list(x), labels)
plt.xlabel("Prompt")
plt.ylabel("Tempo de inferencia (s)")
plt.legend()
plt.tight_layout()
plt.show()